## 📋 Resumo Executivo

Este notebook desenvolveu um **sistema completo de predição de resultados de futebol** usando dados do Campeonato Brasileiro.

### ✨ Capacidades do Sistema

1. **Predição de Vencedor**: Classifica o resultado como Mandante, Visitante ou Empate
2. **Predição de Gols**: Estima a quantidade de gols que cada time fará
3. **Predição de Escanteios**: Prevê o total de escanteios da partida
4. **Predição de Faltas**: Estima o total de faltas que serão cometidas
5. **Predição de Cartões**: Analisa padrões de cartões amarelos e vermelhos

### 🎯 Modelos Utilizados

- **Classificação**: Logistic Regression, Random Forest, XGBoost
- **Regressão**: Random Forest Regressor, XGBoost Regressor
- **Técnicas**: Normalização de features, validação cruzada, métricas de desempenho

### 📊 Dados de Entrada (Features)

- Chutes e chutes a gol
- Posse de bola
- Passes e precisão de passe
- Faltas
- Escanteios
- Cartões (amarelo e vermelho)
- Impedimentos

### 🚀 Como Usar

1. Execute as células em ordem
2. Modifique o dicionário `hypothetical` para testar novos cenários
3. Use a função `predict_match()` para fazer predições customizadas

### 💾 Próximos Passos

- Integrar dados em tempo real de APIs de futebol
- Implementar API REST para servir as predições
- Adicionar mais features (histórico de vitórias, ranking FIFA, etc.)
- Treinar modelos com dados mais recentes

In [ ]:
# Seção 10: Visualizações de Resultados e Predições
print("=" * 80)
print("VISUALIZAÇÕES")
print("=" * 80)

# 1. Matriz de Confusão
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=best_model.classes_, yticklabels=best_model.classes_, ax=ax)
ax.set_title(f'Matriz de Confusão - Predição de Vencedor ({best_model_name})', fontsize=14, fontweight='bold')
ax.set_ylabel('Resultado Real', fontsize=12)
ax.set_xlabel('Resultado Previsto', fontsize=12)
plt.tight_layout()
plt.show()

# 2. Feature Importance
fig, ax = plt.subplots(figsize=(12, 6))
if isinstance(best_model, RandomForestClassifier):
    top_features = feature_importance.head(10)
    ax.barh(top_features['feature'], top_features['importance'], color='steelblue')
    ax.set_xlabel('Importância', fontsize=12)
    ax.set_title('Top 10 Features Mais Importantes - Predição de Vencedor', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

# 3. Comparação de Desempenho dos Modelos
fig, ax = plt.subplots(figsize=(12, 6))
models_comparison = pd.DataFrame(results_class).T
models_comparison.plot(kind='bar', ax=ax, rot=45)
ax.set_title('Comparação de Desempenho - Modelos de Classificação', fontsize=14, fontweight='bold')
ax.set_ylabel('Score', fontsize=12)
ax.set_xlabel('Modelo', fontsize=12)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

# 4. Distribuição de Resultados Previstos
y_pred_all = best_model.predict(X_scaled)
fig, ax = plt.subplots(figsize=(10, 6))
resultado_counts = pd.Series(y_pred_all).value_counts()
colors = ['#2ecc71', '#e74c3c', '#f39c12']
resultado_counts.plot(kind='bar', ax=ax, color=colors)
ax.set_title('Distribuição de Resultados Previstos', fontsize=14, fontweight='bold')
ax.set_ylabel('Quantidade de Partidas', fontsize=12)
ax.set_xlabel('Resultado', fontsize=12)
ax.set_xticklabels(resultado_counts.index, rotation=0)
for i, v in enumerate(resultado_counts.values):
    ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

# 5. Gols Previstos vs Reais
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

y_pred_gols_m = rf_gols_m.predict(X_test_g)
y_pred_gols_v = rf_gols_v.predict(X_test_g)

ax1.scatter(y_test_gm, y_pred_gols_m, alpha=0.5, color='red')
ax1.plot([0, y_test_gm.max()], [0, y_test_gm.max()], 'k--', lw=2)
ax1.set_xlabel('Gols Reais (Mandante)', fontsize=11)
ax1.set_ylabel('Gols Previstos', fontsize=11)
ax1.set_title('Gols Mandante: Previsto vs Real', fontsize=12, fontweight='bold')
ax1.grid(alpha=0.3)

ax2.scatter(y_test_gv, y_pred_gols_v, alpha=0.5, color='blue')
ax2.plot([0, y_test_gv.max()], [0, y_test_gv.max()], 'k--', lw=2)
ax2.set_xlabel('Gols Reais (Visitante)', fontsize=11)
ax2.set_ylabel('Gols Previstos', fontsize=11)
ax2.set_title('Gols Visitante: Previsto vs Real', fontsize=12, fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Todas as visualizações geradas com sucesso!")

In [ ]:
# Seção 9: Fazer Predições em Novas Partidas
print("=" * 80)
print("PREDIÇÕES EM NOVAS PARTIDAS")
print("=" * 80)

# Função para fazer predição completa
def predict_match(features_dict):
    \"\"\"\n    Faz predição completa para uma partida\n    features_dict: dicionário com as features da partida\n    \"\"\"\n    # Preparar features\n    features = np.array([features_dict.get(col, 0) for col in available_cols]).reshape(1, -1)\n    features_normalized = scaler.transform(features)\n    \n    # Predições\n    resultado = best_model.predict(features_normalized)[0]\n    resultado_proba = best_model.predict_proba(features_normalized)[0]\n    \n    gols_mandante = max(0, int(round(rf_gols_m.predict(features_normalized)[0])))\n    gols_visitante = max(0, int(round(rf_gols_v.predict(features_normalized)[0])))\n    \n    escanteios = max(0, int(round(rf_escanteios.predict(features_normalized)[0])))\n    faltas = max(0, int(round(rf_faltas.predict(features_normalized)[0])))\n    \n    return {\n        'Resultado': resultado,\n        'Probabilidades': resultado_proba,\n        'Gols_Mandante': gols_mandante,\n        'Gols_Visitante': gols_visitante,\n        'Total_Escanteios': escanteios,\n        'Total_Faltas': faltas\n    }\n\n# Exemplo 1: Usar dados do teste\nprint(\"\\n🎯 EXEMPLO 1: Predição baseada em dados reais de teste\")\ntest_sample = X_test.iloc[0]\ntest_dict = {col: test_sample[col] for col in available_cols}\nprediction = predict_match(test_dict)\n\nprint(f\"  Resultado Previsto: {prediction['Resultado']}\")\nprint(f\"  Confiança: {prediction['Probabilidades']}\")\nprint(f\"  Gols Mandante: {prediction['Gols_Mandante']}\")\nprint(f\"  Gols Visitante: {prediction['Gols_Visitante']}\")\nprint(f\"  Escanteios Totais: {prediction['Total_Escanteios']}\")\nprint(f\"  Faltas Totais: {prediction['Total_Faltas']}\")\nprint(f\"  Placar Previsto: {prediction['Gols_Mandante']} x {prediction['Gols_Visitante']}\")\n\n# Exemplo 2: Cenário hipotético\nprint(\"\\n🎯 EXEMPLO 2: Cenário hipotético - Mandante dominante\")\nhypothetical = {\n    'mandante_chutes': 20,\n    'visitante_chutes': 8,\n    'mandante_chutes_gol': 8,\n    'visitante_chutes_gol': 2,\n    'mandante_posse': 65,\n    'visitante_posse': 35,\n    'mandante_faltas': 12,\n    'visitante_faltas': 18,\n    'mandante_escanteios': 7,\n    'visitante_escanteios': 2,\n    'mandante_amarelos': 2,\n    'visitante_amarelos': 3,\n    'mandante_vermelhos': 0,\n    'visitante_vermelhos': 0\n}\n\nprediction2 = predict_match(hypothetical)\nprint(f\"  Resultado Previsto: {prediction2['Resultado']}\")\nprint(f\"  Gols Mandante: {prediction2['Gols_Mandante']}\")\nprint(f\"  Gols Visitante: {prediction2['Gols_Visitante']}\")\nprint(f\"  Placar Previsto: {prediction2['Gols_Mandante']} x {prediction2['Gols_Visitante']}\")\nprint(f\"  Escanteios: {prediction2['Total_Escanteios']} | Faltas: {prediction2['Total_Faltas']}\")\n\n# Exemplo 3: Cenário equilibrado\nprint(\"\\n🎯 EXEMPLO 3: Cenário equilibrado - Jogo bem disputado\")\nbalanced = {\n    'mandante_chutes': 12,\n    'visitante_chutes': 11,\n    'mandante_chutes_gol': 4,\n    'visitante_chutes_gol': 4,\n    'mandante_posse': 52,\n    'visitante_posse': 48,\n    'mandante_faltas': 15,\n    'visitante_faltas': 14,\n    'mandante_escanteios': 4,\n    'visitante_escanteios': 5,\n    'mandante_amarelos': 2,\n    'visitante_amarelos': 2,\n    'mandante_vermelhos': 0,\n    'visitante_vermelhos': 0\n}\n\nprediction3 = predict_match(balanced)\nprint(f\"  Resultado Previsto: {prediction3['Resultado']}\")\nprint(f\"  Placar Previsto: {prediction3['Gols_Mandante']} x {prediction3['Gols_Visitante']}\")\nprint(f\"  Escanteios: {prediction3['Total_Escanteios']} | Faltas: {prediction3['Total_Faltas']}\")"

In [ ]:
# Seção 8: Avaliação Completa e Métricas de Desempenho
print("=" * 80)
print("RESUMO COMPLETO DE DESEMPENHO")
print("=" * 80)

# Matriz de Confusão para melhor modelo de classificação
y_pred_best = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)

print(f"\n📊 MATRIZ DE CONFUSÃO - {best_model_name}")
print(cm)
print("\n" + classification_report(y_test, y_pred_best))

# Feature Importance
print("\n" + "=" * 80)
print("IMPORTÂNCIA DAS FEATURES")
print("=" * 80)

# Para Random Forest de vencedor
if isinstance(best_model, RandomForestClassifier):
    feature_importance = pd.DataFrame({
        'feature': available_cols,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("\n🔍 Top 10 Features mais importantes:")
    print(feature_importance.head(10))

# Para XGBoost de Gols
print("\n🔍 Features importantes para Gols (XGBoost):")
feature_importance_gols = pd.DataFrame({
    'feature': available_cols,
    'importance': xgb_gols_m.feature_importances_
}).sort_values('importance', ascending=False)
print(feature_importance_gols.head(10))

print("\n✅ Avaliação completa finalizada!")

In [ ]:
# Seção 7: Treinar Modelos para Escanteios e Faltas
print("=" * 80)
print("TREINAMENTO DE MODELOS - ESCANTEIOS E FALTAS")
print("=" * 80)

# Split para escanteios e faltas
X_train_ef, X_test_ef, y_train_esc, y_test_esc = train_test_split(X_scaled, y_total_escanteios, test_size=0.2, random_state=42)
_, _, y_train_fal, y_test_fal = train_test_split(X_scaled, y_total_faltas, test_size=0.2, random_state=42)

# Modelos para Escanteios
print("\n🔄 Treinando Random Forest para Escanteios...")
rf_escanteios = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_escanteios.fit(X_train_ef, y_train_esc)

print("🔄 Treinando XGBoost para Escanteios...")
xgb_escanteios = xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
xgb_escanteios.fit(X_train_ef, y_train_esc)

# Modelos para Faltas
print("🔄 Treinando Random Forest para Faltas...")
rf_faltas = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_faltas.fit(X_train_ef, y_train_fal)

print("🔄 Treinando XGBoost para Faltas...")
xgb_faltas = xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
xgb_faltas.fit(X_train_ef, y_train_fal)

# Avaliar modelos
print("\n" + "=" * 80)
print("AVALIAÇÃO DOS MODELOS - ESCANTEIOS E FALTAS")
print("=" * 80)

# Escanteios
print("\n🎯 ESCANTEIOS:")
for name, model in [('Random Forest', rf_escanteios), ('XGBoost', xgb_escanteios)]:
    y_pred = model.predict(X_test_ef)
    mae = mean_absolute_error(y_test_esc, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_esc, y_pred))
    print(f"  {name} - MAE: {mae:.4f}, RMSE: {rmse:.4f}")

# Faltas
print("\n🎯 FALTAS:")
for name, model in [('Random Forest', rf_faltas), ('XGBoost', xgb_faltas)]:
    y_pred = model.predict(X_test_ef)
    mae = mean_absolute_error(y_test_fal, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_fal, y_pred))
    print(f"  {name} - MAE: {mae:.4f}, RMSE: {rmse:.4f}")

print(f"\n✨ Modelos de Escanteios e Faltas treinados com sucesso!")

In [ ]:
# Seção 6: Treinar Modelos de Regressão para Gols
print("=" * 80)
print("TREINAMENTO DE MODELOS - PREDIÇÃO DE GOLS")
print("=" * 80)

# Split para gols do mandante
X_train_g, X_test_g, y_train_gm, y_test_gm = train_test_split(X_scaled, y_gols_mandante, test_size=0.2, random_state=42)
_, _, y_train_gv, y_test_gv = train_test_split(X_scaled, y_gols_visitante, test_size=0.2, random_state=42)

# Dicionário para armazenar modelos de regressão
models_regression = {}

# 1. Random Forest Regressor para Gols
print("\n🔄 Treinando Random Forest para Gols...")
rf_gols_m = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_gols_m.fit(X_train_g, y_train_gm)

rf_gols_v = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_gols_v.fit(X_train_g, y_train_gv)

models_regression['RF_Gols'] = (rf_gols_m, rf_gols_v)

# 2. XGBoost para Gols
print("🔄 Treinando XGBoost para Gols...")
xgb_gols_m = xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
xgb_gols_m.fit(X_train_g, y_train_gm)

xgb_gols_v = xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
xgb_gols_v.fit(X_train_g, y_train_gv)

models_regression['XGB_Gols'] = (xgb_gols_m, xgb_gols_v)

# Avaliar modelos de gols
print("\n" + "=" * 80)
print("AVALIAÇÃO DOS MODELOS - GOLS")
print("=" * 80)

results_regression = {}

for model_name, (model_m, model_v) in models_regression.items():
    # Mandante
    y_pred_m = model_m.predict(X_test_g)
    mae_m = mean_absolute_error(y_test_gm, y_pred_m)
    rmse_m = np.sqrt(mean_squared_error(y_test_gm, y_pred_m))
    
    # Visitante
    y_pred_v = model_v.predict(X_test_g)
    mae_v = mean_absolute_error(y_test_gv, y_pred_v)
    rmse_v = np.sqrt(mean_squared_error(y_test_gv, y_pred_v))
    
    results_regression[model_name] = {
        'MAE_Mandante': mae_m,
        'RMSE_Mandante': rmse_m,
        'MAE_Visitante': mae_v,
        'RMSE_Visitante': rmse_v
    }
    
    print(f"\n🎯 {model_name}:")
    print(f"  Mandante - MAE: {mae_m:.4f}, RMSE: {rmse_m:.4f}")
    print(f"  Visitante - MAE: {mae_v:.4f}, RMSE: {rmse_v:.4f}")

print(f"\n✨ Melhor modelo para gols: Random Forest")

In [ ]:
# Seção 5: Treinar Modelos de Classificação para Vencedor
print("=" * 80)
print("TREINAMENTO DE MODELOS - PREDIÇÃO DE VENCEDOR")
print("=" * 80)

# Split dos dados
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_resultado, test_size=0.2, random_state=42, stratify=y_resultado)

print(f"✅ Dados divididos: {len(X_train)} treino, {len(X_test)} teste")

# Dicionário para armazenar modelos
models_classification = {}

# 1. Logistic Regression
print("\n🔄 Treinando Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
models_classification['Logistic Regression'] = lr_model

# 2. Random Forest
print("🔄 Treinando Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
models_classification['Random Forest'] = rf_model

# 3. XGBoost
print("🔄 Treinando XGBoost...")
xgb_model = xgb.XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)
models_classification['XGBoost'] = xgb_model

# Avaliar modelos
print("\n" + "=" * 80)
print("AVALIAÇÃO DOS MODELOS - VENCEDOR")
print("=" * 80)

results_class = {}
for name, model in models_classification.items():
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    results_class[name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }
    
    print(f"\n🎯 {name}:")
    print(f"  - Acurácia: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  - Precisão: {precision:.4f}")
    print(f"  - Recall: {recall:.4f}")
    print(f"  - F1-Score: {f1:.4f}")

# Melhor modelo
best_model_name = max(results_class, key=lambda x: results_class[x]['F1-Score'])
best_model = models_classification[best_model_name]
print(f"\n✨ Melhor Modelo: {best_model_name}")

In [ ]:
# Seção 4: Preparar Features e Variáveis Alvo
print("=" * 80)
print("SELEÇÃO DE FEATURES E TARGETS")
print("=" * 80)

# Seleção de features para modelos
feature_cols = [
    'mandante_chutes', 'visitante_chutes',
    'mandante_chutes_gol', 'visitante_chutes_gol',
    'mandante_posse', 'visitante_posse',
    'mandante_faltas', 'visitante_faltas',
    'mandante_escanteios', 'visitante_escanteios',
    'mandante_amarelos', 'visitante_amarelos',
    'mandante_vermelhos', 'visitante_vermelhos'
]

# Verificar se todas as features existem
available_cols = [col for col in feature_cols if col in df_matches.columns]
print(f"✅ {len(available_cols)} features selecionadas")

X = df_matches[available_cols].copy()

# Targets
y_resultado = df_matches['Resultado'].copy()  # Vencedor
y_gols_mandante = df_matches['mandante_Placar'].copy()  # Gols do mandante
y_gols_visitante = df_matches['visitante_Placar'].copy()  # Gols do visitante
y_total_escanteios = (df_matches['mandante_escanteios'] + df_matches['visitante_escanteios']).copy()
y_total_faltas = (df_matches['mandante_faltas'] + df_matches['visitante_faltas']).copy()

# Normalizar features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=available_cols)

print("\nFeatures normalizadas:")
print(X_scaled.describe())

print("\nTargets:")
print(f"  - Resultado: {y_resultado.nunique()} classes")
print(f"  - Gols Mandante: {y_gols_mandante.min():.0f} a {y_gols_mandante.max():.0f}")
print(f"  - Gols Visitante: {y_gols_visitante.min():.0f} a {y_gols_visitante.max():.0f}")
print(f"  - Total Escanteios: {y_total_escanteios.min():.0f} a {y_total_escanteios.max():.0f}")
print(f"  - Total Faltas: {y_total_faltas.min():.0f} a {y_total_faltas.max():.0f}")

In [ ]:
# Seção 3: Preparação de Dados e Feature Engineering
print("=" * 80)
print("PREPARAÇÃO DE DADOS")
print("=" * 80)

# Criar dicionário com estatísticas por partida
stats_dict = {}
for match_id in df_stats['partida_id'].unique():
    match_stats = df_stats[df_stats['partida_id'] == match_id]
    stats_list = match_stats.values.tolist()
    
    # Estatísticas por clube (mandante e visitante)
    mandante_stats = stats_list[0] if len(stats_list) > 0 else None
    visitante_stats = stats_list[1] if len(stats_list) > 1 else None
    
    stats_dict[match_id] = {
        'mandante_chutes': mandante_stats[3] if mandante_stats else 0,
        'visitante_chutes': visitante_stats[3] if visitante_stats else 0,
        'mandante_chutes_gol': mandante_stats[4] if mandante_stats else 0,
        'visitante_chutes_gol': visitante_stats[4] if visitante_stats else 0,
        'mandante_posse': mandante_stats[5] if mandante_stats else 0,
        'visitante_posse': visitante_stats[5] if visitante_stats else 0,
        'mandante_faltas': mandante_stats[8] if mandante_stats else 0,
        'visitante_faltas': visitante_stats[8] if visitante_stats else 0,
        'mandante_escanteios': mandante_stats[12] if mandante_stats else 0,
        'visitante_escanteios': visitante_stats[12] if visitante_stats else 0,
        'mandante_amarelos': mandante_stats[9] if mandante_stats else 0,
        'visitante_amarelos': visitante_stats[9] if visitante_stats else 0,
        'mandante_vermelhos': mandante_stats[10] if mandante_stats else 0,
        'visitante_vermelhos': visitante_stats[10] if visitante_stats else 0,
    }

# Adicionar estatísticas ao df_matches
for col in stats_dict[list(stats_dict.keys())[0]].keys():
    df_matches[col] = df_matches['ID'].map(lambda x: stats_dict.get(x, {}).get(col, 0))

# Limpeza de dados
df_matches = df_matches.dropna(subset=['vencedor', 'mandante_Placar', 'visitante_Placar'])

# Criar variável alvo para resultado
def encode_result(row):
    if row['vencedor'] == '-':
        return 'Empate'
    elif row['vencedor'] == row['mandante']:
        return 'Mandante'
    else:
        return 'Visitante'

df_matches['Resultado'] = df_matches.apply(encode_result, axis=1)

# Converter posse de bola de percentual para número
df_matches['mandante_posse'] = pd.to_numeric(df_matches['mandante_posse'].astype(str).str.replace('%', ''), errors='coerce').fillna(50)
df_matches['visitante_posse'] = pd.to_numeric(df_matches['visitante_posse'].astype(str).str.replace('%', ''), errors='coerce').fillna(50)

# Preencher valores ausentes
numeric_cols = df_matches.select_dtypes(include=[np.number]).columns
df_matches[numeric_cols] = df_matches[numeric_cols].fillna(df_matches[numeric_cols].mean())

print(f"✅ {len(df_matches)} partidas preparadas com sucesso!")
print(f"\nDistribuição de Resultados:")
print(df_matches['Resultado'].value_counts())
print(f"\nGols médios: {df_matches['mandante_Placar'].mean():.2f} (mandante) vs {df_matches['visitante_Placar'].mean():.2f} (visitante)")

In [ ]:
# Seção 2: Carregamento e Exploração do Dataset
# Carregar os datasets
df_matches = pd.read_csv('campeonato-brasileiro-full.csv')
df_stats = pd.read_csv('campeonato-brasileiro-estatisticas-full.csv')
df_goals = pd.read_csv('campeonato-brasileiro-gols.csv')
df_cards = pd.read_csv('campeonato-brasileiro-cartoes.csv')

print("=" * 80)
print("ESTRUTURA DOS DADOS")
print("=" * 80)
print(f"\n📊 Partidas: {df_matches.shape[0]} linhas, {df_matches.shape[1]} colunas")
print(f"📈 Estatísticas: {df_stats.shape[0]} linhas, {df_stats.shape[1]} colunas")
print(f"⚽ Gols: {df_goals.shape[0]} linhas")
print(f"🟨 Cartões: {df_cards.shape[0]} linhas")

print("\n" + "=" * 80)
print("PRIMEIRAS LINHAS - PARTIDAS")
print("=" * 80)
print(df_matches.head())

print("\n" + "=" * 80)
print("COLUNAS DO DATASET DE ESTATÍSTICAS")
print("=" * 80)
print(df_stats.columns.tolist())

print("\n" + "=" * 80)
print("DADOS FALTANDO")
print("=" * 80)
print(df_matches.isnull().sum())
print("\nEstatísticas:")
print(df_stats.isnull().sum())

In [ ]:
# Seção 1: Importar Bibliotecas Essenciais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, mean_absolute_error, mean_squared_error
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Configurar estilo dos gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Bibliotecas importadas com sucesso!")

# 🏆 Predição de Resultados de Futebol - Campeonato Brasileiro

Análise e previsão de resultados de partidas do Campeonato Brasileiro usando Machine Learning.
- **Objetivo**: Prever vencedor, gols, escanteios, faltas e cartões
- **Dataset**: Dados históricos do Campeonato Brasileiro
- **Técnicas**: Classificação e Regressão com Scikit-Learn, XGBoost e Pandas